# 2: Data Storage — SQL + NoSQL + Data Lake
**Pipeline Stage:** Structured, queryable storage with integrity checks.

This notebook demonstrates:
- **SQLite** as a relational store (SQL layer)
- **TinyDB** as a lightweight NoSQL document store
- **Parquet files** as a columnar Data Lake format
- Data integrity checks (primary key deduplication, type enforcement, checksums)
- Basic access-control simulation

In [11]:
!pip install pandas tinydb pyarrow

In [12]:
import pandas as pd
import sqlite3
import hashlib
import json
import os
from pathlib import Path
from datetime import datetime
from tinydb import TinyDB, Query

BASE_DIR   = Path('..').resolve()
OUTPUT_DIR = BASE_DIR / 'outputs'
DB_PATH    = OUTPUT_DIR / 'pipeline.db'
LAKE_DIR   = OUTPUT_DIR / 'data_lake'
LAKE_DIR.mkdir(exist_ok=True)

print("Paths ready.")

Paths ready.


## 2.1  Read Raw Data from Data Ingestion Notebook

In [13]:
conn = sqlite3.connect(DB_PATH)
df_raw = pd.read_sql('SELECT * FROM raw_sessions', conn)
conn.close()
print(f"Loaded {len(df_raw):,} rows from raw_sessions")
df_raw.head(2)

Loaded 12,530 rows from raw_sessions


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue,ingestion_source,ingestion_timestamp,row_id
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,...,1,1,1,1,Returning_Visitor,0,0,csv_batch,2026-05-03T20:36:01.543693,0
1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,...,2,2,1,2,Returning_Visitor,0,0,csv_batch,2026-05-03T20:36:01.543693,1


## 2.2  Data Integrity Checks

In [14]:
def run_integrity_checks(df: pd.DataFrame) -> pd.DataFrame:
    """Run integrity checks and return a cleaned DataFrame with a checksum."""
    original_len = len(df)

    # 1. Deduplicate on all feature columns (not metadata cols)
    feature_cols = [c for c in df.columns if c not in
                    ['row_id','ingestion_source','ingestion_timestamp','checksum']]
    df = df.drop_duplicates(subset=feature_cols).copy()
    print(f"[INTEGRITY] Duplicates removed: {original_len - len(df)}")

    # 2. Type enforcement
    df['Revenue'] = df['Revenue'].astype(int)
    df['Weekend'] = df['Weekend'].astype(int)
    df['BounceRates'] = pd.to_numeric(df['BounceRates'], errors='coerce').fillna(0)
    df['ExitRates']   = pd.to_numeric(df['ExitRates'],   errors='coerce').fillna(0)

    # 3. Range validation
    assert df['BounceRates'].between(0,1).all(), "BounceRates out of [0,1] range"
    assert df['ExitRates'].between(0,1).all(),   "ExitRates out of [0,1] range"
    print("[INTEGRITY] Range checks: PASSED")

    # 4. Add row-level SHA-256 checksum for tamper detection
    def row_checksum(row):
        payload = '|'.join(str(row[c]) for c in feature_cols)
        return hashlib.sha256(payload.encode()).hexdigest()[:16]

    df['checksum'] = df.apply(row_checksum, axis=1)
    print(f"[INTEGRITY] Checksums added. Final rows: {len(df):,}")
    return df

df_clean = run_integrity_checks(df_raw)
df_clean[['Revenue','BounceRates','ExitRates','checksum']].head(3)

[INTEGRITY] Duplicates removed: 125
[INTEGRITY] Range checks: PASSED
[INTEGRITY] Checksums added. Final rows: 12,405


,Revenue,BounceRates,ExitRates,checksum
0,0,0.2,0.2,bca403819024410e
1,0,0.0,0.1,a5d8ad33d1f82082
2,0,0.2,0.2,2d8a488efdd01178


## 2.3  SQL Storage — SQLite Structured Tables

In [15]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Create a clean sessions table with a proper schema
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sessions (
        id                      INTEGER PRIMARY KEY AUTOINCREMENT,
        row_id                  INTEGER,
        Administrative          INTEGER,
        Administrative_Duration REAL,
        Informational           INTEGER,
        Informational_Duration  REAL,
        ProductRelated          INTEGER,
        ProductRelated_Duration REAL,
        BounceRates             REAL,
        ExitRates               REAL,
        PageValues              REAL,
        SpecialDay              REAL,
        Month                   TEXT,
        OperatingSystems        INTEGER,
        Browser                 INTEGER,
        Region                  INTEGER,
        TrafficType             INTEGER,
        VisitorType             TEXT,
        Weekend                 INTEGER,
        Revenue                 INTEGER,
        ingestion_source        TEXT,
        ingestion_timestamp     TEXT,
        checksum                TEXT
    )
''')
conn.commit()

# Write clean data
store_cols = [c for c in df_clean.columns if c in [
    'row_id','Administrative','Administrative_Duration','Informational',
    'Informational_Duration','ProductRelated','ProductRelated_Duration',
    'BounceRates','ExitRates','PageValues','SpecialDay','Month',
    'OperatingSystems','Browser','Region','TrafficType','VisitorType',
    'Weekend','Revenue','ingestion_source','ingestion_timestamp','checksum'
]]
df_clean[store_cols].to_sql('sessions', conn, if_exists='replace', index=False)

# Revenue summary view
cursor.execute('''
    CREATE VIEW IF NOT EXISTS revenue_summary AS
    SELECT Month, VisitorType,
           COUNT(*) as sessions,
           SUM(Revenue) as conversions,
           ROUND(100.0*SUM(Revenue)/COUNT(*), 2) as conversion_rate_pct
    FROM sessions
    GROUP BY Month, VisitorType
''')
conn.commit()
conn.close()
print("SQL tables and views created.")

SQL tables and views created.


In [16]:
# Quick query to verify
conn = sqlite3.connect(DB_PATH)
result = pd.read_sql('SELECT * FROM revenue_summary ORDER BY conversion_rate_pct DESC LIMIT 8', conn)
conn.close()
result

,Month,VisitorType,sessions,conversions,conversion_rate_pct
0,Apr,New_Visitor,9,4,44.44
1,Jan,Returning_Visitor,3,1,33.33
2,Nov,New_Visitor,426,129,30.28
3,Aug,New_Visitor,78,23,29.49
4,Apr,Returning_Visitor,7,2,28.57
5,May,New_Visitor,325,89,27.38
6,Nov,Returning_Visitor,2545,629,24.72
7,Sep,New_Visitor,115,28,24.35


## 2.4  NoSQL Storage — TinyDB (Document Store)

In [17]:
nosql_path = OUTPUT_DIR / 'nosql_sessions.json'
db_nosql = TinyDB(nosql_path)
db_nosql.drop_tables()  # clear previous runs

sessions_table = db_nosql.table('sessions')

# Store a sample of 500 records as JSON documents
sample = df_clean.head(500).to_dict(orient='records')
sessions_table.insert_multiple(sample)

print(f"NoSQL: inserted {len(sessions_table)} documents")

# Example document query: revenue sessions from November
Session = Query()
nov_revenue = sessions_table.search((Session.Month == 'Nov') & (Session.Revenue == 1))
print(f"November converting sessions: {len(nov_revenue)}")

NoSQL: inserted 500 documents
November converting sessions: 0


## 2.5  Data Lake — Parquet (Columnar Storage)

In [18]:
# Partition data lake by Month (mimics Hive partitioning on S3/HDFS)
for month, group in df_clean.groupby('Month'):
    partition_dir = LAKE_DIR / f'month={month}'
    partition_dir.mkdir(exist_ok=True)
    parquet_file = partition_dir / 'data.parquet'
    group.to_parquet(parquet_file, index=False)

# List partitions
partitions = list(LAKE_DIR.glob('month=*/data.parquet'))
print(f"Data Lake partitions created: {len(partitions)}")
for p in sorted(partitions):
    size_kb = p.stat().st_size // 1024
    print(f"  {p.parent.name}  →  {size_kb} KB")

Data Lake partitions created: 12
  month=Apr  →  15 KB
  month=Aug  →  39 KB
  month=Dec  →  98 KB
  month=Feb  →  22 KB
  month=Jan  →  14 KB
  month=Jul  →  39 KB
  month=June  →  30 KB
  month=Mar  →  99 KB
  month=May  →  174 KB
  month=Nov  →  171 KB
  month=Oct  →  45 KB
  month=Sep  →  39 KB


In [19]:
# Read back a single partition
import pyarrow.parquet as pq
sample_partition = sorted(LAKE_DIR.glob('month=*/data.parquet'))[0]
df_lake = pq.read_table(sample_partition).to_pandas()
print(f"Read back {len(df_lake)} rows from partition: {sample_partition.parent.name}")
df_lake.head(2)

Read back 20 rows from partition: month=Apr


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Region,TrafficType,VisitorType,Weekend,Revenue,ingestion_source,ingestion_timestamp,row_id,checksum,month
0,15,35.55,2,26.11,25,1350.32,0.0688,0.1966,11.16,0.0,...,4,5,Other,1,0,kafka_stream,2026-05-03T20:36:01.658556,13,e0a37119ca93fdb6,Apr
1,9,76.03,1,163.69,27,2523.01,0.0558,0.0026,62.85,0.0,...,5,20,Returning_Visitor,1,0,kafka_stream,2026-05-03T20:36:01.658556,27,d8c8efe7fc17ccdf,Apr


## 2.6  Access Control Simulation

In [20]:
# Simulate role-based access by masking sensitive columns for non-admin roles
ROLES = {
    'admin':    None,               # full access
    'analyst':  ['Revenue','Month','VisitorType','BounceRates','ExitRates','PageValues'],
    'viewer':   ['Month','VisitorType','Revenue']
}

def fetch_for_role(df: pd.DataFrame, role: str) -> pd.DataFrame:
    allowed = ROLES.get(role)
    if allowed is None:
        return df
    return df[[c for c in df.columns if c in allowed]]

for role in ['admin','analyst','viewer']:
    visible = fetch_for_role(df_clean, role)
    print(f"Role '{role}': {visible.shape[1]} columns visible")

Role 'admin': 22 columns visible
Role 'analyst': 6 columns visible
Role 'viewer': 3 columns visible


## Summary
| Storage Layer | Technology | Location |
|---|---|---|
| Relational (SQL) | SQLite | `outputs/pipeline.db` |
| NoSQL (Document) | TinyDB | `outputs/nosql_sessions.json` |
| Data Lake (Columnar) | Parquet (partitioned by Month) | `outputs/data_lake/` |